# Linear Derivatives and Carry

Module: Derivatives

## Lesson summary

Forwards, futures, and swaps are the linear foundation of derivatives pricing. Their values are driven by no-arbitrage carry, discounting, and curve consistency rather than by payoff convexity. This lesson connects spot price, dividend yield, interest rates, mark-to-market value, and par swap rates.

The examples use deterministic inputs. Mexican TIIE, CETES, and FX series can be connected later through the existing data layer, but this notebook does not require credentials or network access.

## Learning objectives

By the end of this lesson, students should be able to:

- compute a forward price under continuous cost of carry;
- mark an existing forward contract to market;
- explain when futures and forwards differ conceptually;
- compute a par swap rate from discount factors and accrual fractions;
- interpret carry assumptions for FX and equity-linked derivatives.

## Python setup

In [ ]:
import numpy as np
import pandas as pd

from src.derivatives import forward_price, forward_value, par_swap_rate

## Forward price with continuous carry

For an asset with spot price $S_0$, risk-free rate $r$, dividend or convenience yield $q$, and maturity $T$:

$$
F(0,T) = S_0 e^{(r-q)T}.
$$

In [ ]:
spot = 100.0
rate = 0.095
maturity = 0.5
dividend_yield = 0.025

forward = forward_price(spot, rate, maturity, dividend_yield)
forward

## Mark-to-market value

If a forward was originally struck at $K$, the value to the long side before maturity is:

$$
V_t = S_t e^{-q(T-t)} - K e^{-r(T-t)}.
$$

In [ ]:
strikes = [95, 100, forward, 110]

pd.DataFrame(
    {
        "strike": strikes,
        "long_forward_value": [
            forward_value(spot, strike, rate, maturity, dividend_yield)
            for strike in strikes
        ],
    }
)

## Carry scenarios

Carry explains why forward prices can be above or below spot.

In [ ]:
scenarios = pd.DataFrame(
    {
        "scenario": ["high domestic rate", "high dividend yield", "flat carry"],
        "rate": [0.10, 0.04, 0.06],
        "dividend_yield": [0.02, 0.08, 0.06],
    }
)
scenarios["forward_price"] = [
    forward_price(spot, r, 1.0, q)
    for r, q in zip(scenarios["rate"], scenarios["dividend_yield"])
]
scenarios["forward_minus_spot"] = scenarios["forward_price"] - spot
scenarios

## Par swap rate

A fixed-for-floating interest-rate swap can be valued as a fixed leg against a floating leg. At inception, the par fixed rate sets the swap value to zero:

$$
R_{swap} =
\frac{1 - D(0,T_n)}
{\sum_{i=1}^{n}\tau_i D(0,T_i)}.
$$

In [ ]:
payment_dates = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
zero_rates = np.array([0.087, 0.085, 0.083, 0.082, 0.081, 0.080])
discount_factors = np.exp(-zero_rates * payment_dates)
accruals = np.full_like(payment_dates, 0.5)

swap_table = pd.DataFrame(
    {
        "payment_date": payment_dates,
        "zero_rate": zero_rates,
        "discount_factor": discount_factors,
        "accrual": accruals,
    }
)
swap_table

In [ ]:
par_rate = par_swap_rate(discount_factors, accruals)
par_rate

## Interpretation notes

| Contract | Core pricing input | Main classroom risk |
| --- | --- | --- |
| Forward | Spot, carry, maturity | Using an inconsistent dividend or funding assumption |
| Future | Forward logic plus margining | Ignoring stochastic-rate convexity adjustment |
| Swap | Discount curve and accrual factors | Mixing discount and projection curves without documentation |

## Model limitations

- Carry formulas depend on clean inputs for funding, income, storage, convenience yield, and collateral assumptions.
- Forward and futures prices can differ when rates, margins, taxes, or settlement mechanics matter.
- Par swap calculations require a reliable discount curve and cash-flow conventions.